[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajitpanday80/ai-learn/blob/main/notebooks/ft-what-is.ipynb)

# 🎯 What is Fine-tuning?

**Fine-tuning** means taking a model that was already **pre-trained** on a huge, general dataset and training it a little more on a **small, task-specific dataset**.

Think of it like hiring an experienced chef and teaching them your restaurant's menu. You don't teach them to cook from scratch. You only teach them what's specific to your restaurant.

| | Training from scratch | Fine-tuning |
|---|---|---|
| Starting weights | Random | Pre-trained (already knows language) |
| Data needed | Millions of examples | Dozens to thousands |
| Compute | Days/weeks on many GPUs | Minutes on one GPU |
| Learning rate | Larger | Small (so you don't erase existing knowledge) |

### In this lesson you will:
1. Load a pre-trained language model (`distilbert-base-uncased`) that has **never seen a sentiment label**
2. Measure how badly it does on sentiment classification **before** fine-tuning
3. Fine-tune it on only **24 sentences** and watch the accuracy jump
4. Compare **full fine-tuning** with **training only the head** (a frozen base)

> ⚡ **Enable a GPU:** In Colab, go to **Runtime → Change runtime type → GPU (T4)**. The notebook also runs on CPU, just more slowly.

In [ ]:
!pip install -q transformers torch

import random
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()  # hide the 'some weights are newly initialized' warning
plt.style.use('dark_background')

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cpu':
    print('Tip: Runtime > Change runtime type > GPU for faster training')

## 📦 Step 1: A tiny task-specific dataset and a pre-trained model

`distilbert-base-uncased` was pre-trained to predict masked words across Wikipedia and books. It has learned a lot about English, but it has **never been told what 'positive' or 'negative' means**.

When we load it with `AutoModelForSequenceClassification`, Hugging Face adds a new **classification head**: a small linear layer with **random weights** on top of the pre-trained **body**. Before fine-tuning, that random head produces roughly coin-flip predictions.

```
[ Pre-trained DistilBERT body (66M params) ]  →  [ New random head (~600k params) ]  →  positive / negative
```

In [ ]:
# ✏️ EXPERIMENT: add your own sentences! (label 1 = positive, 0 = negative)
train_data = [
    ('I absolutely loved this movie, it was fantastic', 1),
    ('What a wonderful experience, highly recommend', 1),
    ('The food was delicious and the staff were friendly', 1),
    ('Best purchase I have made all year', 1),
    ('This book was a joy to read from start to finish', 1),
    ('Great quality and super fast delivery', 1),
    ('The concert was amazing, what an energy', 1),
    ('I am so happy with how this turned out', 1),
    ('Brilliant acting and a beautiful story', 1),
    ('The hotel was clean, cozy and comfortable', 1),
    ('Everything worked perfectly out of the box', 1),
    ('A delightful little cafe with lovely coffee', 1),
    ('This was a complete waste of money', 0),
    ('Terrible service, I will never come back', 0),
    ('The movie was boring and far too long', 0),
    ('It broke after two days, very disappointed', 0),
    ('The food was cold and tasted awful', 0),
    ('Worst customer support I have ever dealt with', 0),
    ('I regret buying this, it is useless', 0),
    ('The plot made no sense and the acting was bad', 0),
    ('The room was dirty and smelled horrible', 0),
    ('Shipping took forever and the box was damaged', 0),
    ('A dull, lifeless and forgettable book', 0),
    ('Nothing worked and the instructions were confusing', 0),
]

test_data = [
    ('An incredible film that moved me to tears', 1),
    ('Friendly people and a great atmosphere', 1),
    ('I would happily buy this again', 1),
    ('Such a pleasant surprise, really impressive', 1),
    ('The sound quality is excellent', 1),
    ('Our waiter was kind and attentive', 1),
    ('Awful experience from beginning to end', 0),
    ('The battery died within an hour, rubbish', 0),
    ('I fell asleep halfway through, so boring', 0),
    ('Rude staff and overpriced drinks', 0),
    ('The product arrived broken and unusable', 0),
    ('I want my money back, this is terrible', 0),
]

train_texts, train_labels = zip(*train_data)
test_texts, test_labels = zip(*test_data)
train_texts, train_labels = list(train_texts), list(train_labels)
test_texts, test_labels = list(test_texts), list(test_labels)

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def fresh_model(seed=42):
    set_seed(seed)
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

@torch.no_grad()
def evaluate(model, texts, labels):
    model.eval()
    enc = tokenizer(texts, padding=True, truncation=True, return_tensors='pt').to(device)
    preds = model(**enc).logits.argmax(-1).cpu()
    return (preds == torch.tensor(labels)).float().mean().item()

@torch.no_grad()
def predict(model, text):
    model.eval()
    enc = tokenizer(text, return_tensors='pt').to(device)
    probs = torch.softmax(model(**enc).logits, dim=-1)[0].cpu()
    label = 'POSITIVE' if probs[1] > probs[0] else 'NEGATIVE'
    return label, probs.max().item()

base_model = fresh_model()
before_acc = evaluate(base_model, test_texts, test_labels)
print(f'Train examples: {len(train_texts)} | Test examples: {len(test_texts)}')
print(f'Test accuracy BEFORE fine-tuning: {before_acc:.0%}  (about chance level)')

In [ ]:
def fine_tune(model, epochs=6, lr=5e-5, batch_size=8, freeze_base=False, verbose=True):
    '''Minimal PyTorch fine-tuning loop. Returns loss/accuracy history.'''
    if freeze_base:
        for p in model.distilbert.parameters():
            p.requires_grad = False

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr)
    history = {'step_loss': [], 'test_acc': [evaluate(model, test_texts, test_labels)]}

    for epoch in range(epochs):
        model.train()
        idx = list(range(len(train_texts)))
        random.shuffle(idx)
        for i in range(0, len(idx), batch_size):
            batch = idx[i:i + batch_size]
            enc = tokenizer([train_texts[j] for j in batch], padding=True,
                            truncation=True, return_tensors='pt').to(device)
            y = torch.tensor([train_labels[j] for j in batch]).to(device)
            loss = model(**enc, labels=y).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            history['step_loss'].append(loss.item())
        acc = evaluate(model, test_texts, test_labels)
        history['test_acc'].append(acc)
        if verbose:
            print(f'Epoch {epoch + 1}/{epochs}  loss={loss.item():.4f}  test_acc={acc:.0%}')
    return history

# ✏️ EXPERIMENT: try LEARNING_RATE = 1e-3 (too high, may erase knowledge) or 1e-6 (too low)
EPOCHS = 6
LEARNING_RATE = 5e-5
BATCH_SIZE = 8

model = fresh_model()
hist = fine_tune(model, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist['step_loss'], color='#4FC3F7', marker='o', markersize=3)
axes[0].set_title('Training loss per step')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Cross-entropy loss')
axes[0].grid(alpha=0.2)

axes[1].plot(range(len(hist['test_acc'])), hist['test_acc'], color='#FFB74D', marker='o')
axes[1].axhline(0.5, color='gray', linestyle='--', label='Chance (50%)')
axes[1].set_title('Test accuracy by epoch (epoch 0 = before fine-tuning)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()

# ✏️ EXPERIMENT: write your own sentences and compare the models before and after fine-tuning
my_sentences = [
    'This lesson was really helpful!',
    'I hated every minute of it.',
    'The weather ruined our picnic.',
    'Not bad at all, actually quite good.',
]
print('\nText'.ljust(42), '| Before'.ljust(22), '| After')
for s in my_sentences:
    b_label, b_conf = predict(base_model, s)
    a_label, a_conf = predict(model, s)
    print(f'{s[:40]:<40} | {b_label} ({b_conf:.0%})'.ljust(64), f'| {a_label} ({a_conf:.0%})')

## 🧊 Step 2: Full fine-tuning vs. training only the head

There are two main strategies:

- **Full fine-tuning:** update *every* weight in the model. This is the most flexible option, but it uses the most memory and has the highest risk of *catastrophic forgetting*.
- **Frozen base (feature extraction):** freeze the pre-trained body and train **only the new head**. It's much cheaper, but the model's internal representations can't adapt to your task.

Modern methods such as **LoRA** and other **PEFT** (Parameter-Efficient Fine-Tuning) techniques fall between these two. They train a small number of *extra* parameters inside the body. You'll cover them in later lessons.

The next cell counts the trainable parameters for each strategy and compares their accuracy.

> 💡 A frozen head usually needs a **higher learning rate** (e.g. `1e-3`) because it starts from random weights and only has a few parameters to adjust.

In [ ]:
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ✏️ EXPERIMENT: change these settings and rerun the comparison
strategies = {
    'Full fine-tune':  dict(freeze_base=False, lr=5e-5),
    'Head only (frozen)': dict(freeze_base=True, lr=1e-3),
}
COMPARE_EPOCHS = 6

results = {}
for name, cfg in strategies.items():
    print(f'--- {name} ---')
    m = fresh_model()
    h = fine_tune(m, epochs=COMPARE_EPOCHS, lr=cfg['lr'], freeze_base=cfg['freeze_base'], verbose=False)
    results[name] = {'params': count_trainable(m), 'acc': h['test_acc']}
    print(f'Trainable params: {results[name]["params"]:,} | final test acc: {h["test_acc"][-1]:.0%}')
    del m
    if device == 'cuda':
        torch.cuda.empty_cache()

colors = ['#4FC3F7', '#F06292', '#81C784', '#FFB74D']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(results)

bars = axes[0].bar(names, [results[n]['params'] for n in names], color=colors[:len(names)])
axes[0].set_yscale('log')
axes[0].set_title('Trainable parameters (log scale)')
for b, n in zip(bars, names):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height(), f'{results[n]["params"]:,}',
                 ha='center', va='bottom', fontsize=9)

for c, n in zip(colors, names):
    axes[1].plot(results[n]['acc'], marker='o', color=c, label=n)
axes[1].axhline(0.5, color='gray', linestyle='--', label='Chance')
axes[1].set_title('Test accuracy by epoch')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.05); axes[1].legend(); axes[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()

## ✅ Key takeaways

- **Fine-tuning = pre-trained knowledge + a small amount of task data.** With only 24 examples, the model went from about chance level to strong accuracy. That works because it already understood language.
- **Use a small learning rate for full fine-tuning** (around `1e-5` to `5e-5`). If it's too high, you can overwrite what the model learned in pre-training (*catastrophic forgetting*).
- **You can choose how much of the model to train.** Full fine-tuning is the most flexible. A frozen body is the cheapest. PEFT methods like LoRA sit in between.

### 🧪 Try these experiments
1. Set `LEARNING_RATE = 1e-3` for full fine-tuning. Does the training become unstable?
2. Shrink the training set to 6 examples. How few examples still work?
3. Add tricky test sentences with sarcasm or negation ("not bad", "yeah, great, it broke again").
4. Add a third strategy to `strategies` with `freeze_base=True, lr=5e-5`. Why does it learn so slowly?
5. Change `MODEL_NAME` to `'bert-base-uncased'` or `'roberta-base'`. Note that `freeze_base` uses `model.distilbert`, so change it to `model.base_model` for these models.

**Next up:** preparing real datasets and fine-tuning with the Hugging Face `Trainer` API.